# goldBERT — Fine-tuning FinBERT for Gold-News Sentiment

Fine-tune [`ProsusAI/finbert`](https://huggingface.co/ProsusAI/finbert) on the gold-news dataset
(`data/merged_normalized_no_neutral.csv`, exported from `../data_preparation/`).

- **Input**: the headline (`News`) combined with the `article_body`.
- **Target**: `Price Sentiment` — 2 classes: `positive`, `negative`.
- **Dataset**: built by `data_preparation/scripts/run_pipeline.py` (neutral/none rows removed upstream).

Pipeline: load & clean → build text/labels → stratified train/val/test split →
tokenize → fine-tune → evaluate → save.

In [1]:
import os
import numpy as np
import pandas as pd
import torch

from datasets import Dataset
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, f1_score, precision_recall_fscore_support, classification_report
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    DataCollatorWithPadding,
    Trainer,
    TrainingArguments,
    set_seed,
)

# ---------------------------------------------------------------- config
SEED = 42
MODEL_NAME = "ProsusAI/finbert"
DATA_PATH = "data/merged_normalized_no_neutral.csv"
OUTPUT_DIR = "goldbert-finetuned"

TEXT_COLS = ["News", "article_body"]   # headline + body -> model input
LABEL_COL = "Price Sentiment"
LABELS = ["positive", "negative"]

MAX_LENGTH = 256          # tokens (headline+body); raise to 512 for more body context
TRAIN_BATCH = 16
EVAL_BATCH = 32
EPOCHS = 3
LR = 2e-5
WEIGHT_DECAY = 0.01

set_seed(SEED)

label2id = {l: i for i, l in enumerate(LABELS)}
id2label = {i: l for l, i in label2id.items()}

device = "cuda" if torch.cuda.is_available() else ("mps" if getattr(torch.backends, "mps", None) and torch.backends.mps.is_available() else "cpu")
print("Using device:", device)
print("Labels:", label2id)

/home/tarekichalalen/Desktop/Trading_bot/gold_env/lib64/python3.14/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Using device: cuda
Labels: {'positive': 0, 'negative': 1}


## 1. Load data & build inputs/labels

The `article_body` is sometimes missing or the literal string `"empty"`; in those
cases we fall back to just the headline. This split only contains `positive` and
`negative` labels.

In [2]:
df = pd.read_csv(DATA_PATH, dtype=str)
print("Raw rows:", len(df))


def clean_body(value: str) -> str:
    if not isinstance(value, str):
        return ""
    v = value.strip()
    if v.lower() == "empty":
        return ""
    return v


headline = df["News"].fillna("").str.strip()
body = df["article_body"].map(clean_body)

df = df.assign(headline=headline, body=body)

df = df[df["headline"].str.len() > 0]
df = df[df[LABEL_COL].isin(LABELS)].reset_index(drop=True)
df["label"] = df[LABEL_COL].map(label2id).astype(int)

print("Usable rows:", len(df))
print("\nLabel distribution:")
print(df[LABEL_COL].value_counts())
print("\nRows with a non-empty body:", int((df["body"].str.len() > 0).sum()))

Raw rows: 8226
Usable rows: 8226

Label distribution:
Price Sentiment
positive    4412
negative    3814
Name: count, dtype: int64

Rows with a non-empty body: 3908


In [3]:
# Stratified 80/10/10 train/val/test split.
train_df, temp_df = train_test_split(
    df, test_size=0.20, random_state=SEED, stratify=df["label"]
)
val_df, test_df = train_test_split(
    temp_df, test_size=0.50, random_state=SEED, stratify=temp_df["label"]
)

for name, part in [("train", train_df), ("val", val_df), ("test", test_df)]:
    print(f"{name:5s}: {len(part):5d}")

# Class weights to counter imbalance between positive and negative.
counts = train_df["label"].value_counts().sort_index().to_numpy()
class_weights = torch.tensor((counts.sum() / (len(counts) * counts)), dtype=torch.float)
print("\nClass weights:", {id2label[i]: round(float(w), 3) for i, w in enumerate(class_weights)})

train:  6580
val  :   823
test :   823

Class weights: {'positive': 0.932, 'negative': 1.078}


## 2. Tokenize

We feed the headline and body as a sentence pair `(headline, body)`. The tokenizer
inserts `[SEP]` between them and truncates the longer body first (`only_second`).

In [4]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)


def to_dataset(frame: pd.DataFrame) -> Dataset:
    return Dataset.from_dict(
        {
            "headline": frame["headline"].tolist(),
            "body": frame["body"].tolist(),
            "labels": frame["label"].tolist(),
        }
    )


def tokenize_batch(batch):
    return tokenizer(
        batch["headline"],
        batch["body"],
        truncation="only_second",
        max_length=MAX_LENGTH,
    )


keep_cols = ["input_ids", "attention_mask", "labels"]
tokenized = {}
for name, frame in [("train", train_df), ("val", val_df), ("test", test_df)]:
    ds = to_dataset(frame).map(tokenize_batch, batched=True)
    ds = ds.remove_columns([c for c in ds.column_names if c not in keep_cols])
    tokenized[name] = ds

data_collator = DataCollatorWithPadding(tokenizer=tokenizer)
print(tokenized["train"])
print("Example token length:", len(tokenized["train"][0]["input_ids"]))

Map: 100%|█████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 823/823 [00:00<00:00, 11674.33 examples/s]

Dataset({
    features: ['labels', 'input_ids', 'attention_mask'],
    num_rows: 6580
})
Example token length: 26


## 3. Model, metrics & weighted loss

We load FinBERT with a fresh 2-class head (`positive`, `negative`) and use a
class-weighted cross-entropy loss.

In [5]:
model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=len(LABELS),
    id2label=id2label,
    label2id=label2id,
    ignore_mismatched_sizes=True,  # FinBERT head is 3-class; reinit for 2 classes
)


def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)
    acc = accuracy_score(labels, preds)
    f1_macro = f1_score(labels, preds, average="macro")
    f1_weighted = f1_score(labels, preds, average="weighted")
    return {"accuracy": acc, "f1_macro": f1_macro, "f1_weighted": f1_weighted}


class WeightedTrainer(Trainer):
    """Trainer with class-weighted cross-entropy loss."""

    def __init__(self, *args, class_weights=None, **kwargs):
        super().__init__(*args, **kwargs)
        self._class_weights = class_weights

    def compute_loss(self, model, inputs, return_outputs=False, **kwargs):
        labels = inputs.pop("labels")
        outputs = model(**inputs)
        weights = None
        if self._class_weights is not None:
            weights = self._class_weights.to(outputs.logits.device)
        loss_fct = torch.nn.CrossEntropyLoss(weight=weights)
        loss = loss_fct(outputs.logits.view(-1, len(LABELS)), labels.view(-1))
        return (loss, outputs) if return_outputs else loss

[transformers] You passed `num_labels=2` which is incompatible to the `id2label` map of length `3`.
Loading weights: 100%|████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 201/201 [00:00<00:00, 71530.21it/s]
[transformers] BertForSequenceClassification LOAD REPORT from: ProsusAI/finbert
Key               | Status   |                                                                                       
------------------+----------+---------------------------------------------------------------------------------------
classifier.bias   | MISMATCH | Reinit due to size mismatch - ckpt: torch.Size([3]) vs model:torch.Size([2])          
classifier.weight | MISMATCH | Reinit due to size mismatch - ckpt: torch.Size([3, 768]) vs model:torch.Size([2, 768])

Notes:
- MISMATCH:	ckpt weights were loaded, but they did not match the original empty weight shapes.


## 4. Train

The best checkpoint (by validation macro-F1) is reloaded at the end of training.

In [6]:
use_fp16 = device == "cuda"

training_args = TrainingArguments(
    output_dir=OUTPUT_DIR,
    eval_strategy="epoch",
    save_strategy="epoch",
    learning_rate=LR,
    per_device_train_batch_size=TRAIN_BATCH,
    per_device_eval_batch_size=EVAL_BATCH,
    num_train_epochs=EPOCHS,
    weight_decay=WEIGHT_DECAY,
    warmup_ratio=0.1,
    logging_steps=50,
    load_best_model_at_end=True,
    metric_for_best_model="f1_macro",
    greater_is_better=True,
    save_total_limit=2,
    fp16=use_fp16,
    report_to="none",
    seed=SEED,
)

trainer = WeightedTrainer(
    model=model,
    args=training_args,
    train_dataset=tokenized["train"],
    eval_dataset=tokenized["val"],
    processing_class=tokenizer,
    data_collator=data_collator,
    compute_metrics=compute_metrics,
    class_weights=class_weights,
)

train_result = trainer.train()
print(train_result.metrics)

[transformers] warmup_ratio is deprecated and will be removed in v5.2. Use `warmup_steps` instead.


Epoch,Training Loss,Validation Loss,Accuracy,F1 Macro,F1 Weighted
1,0.143416,0.104817,0.963548,0.963243,0.963483
2,0.068243,0.091273,0.974484,0.974308,0.974460
3,0.037463,0.126746,0.973269,0.973057,0.973228


Writing model shards: 100%|██████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 1/1 [00:00<00:00,  3.34it/s]


{'train_runtime': 190.6093, 'train_samples_per_second': 103.563, 'train_steps_per_second': 6.484, 'total_flos': 2589419784656160.0, 'train_loss': 0.12049699774837802, 'epoch': 3.0}


## 5. Evaluate on the held-out test set

In [7]:
test_metrics = trainer.evaluate(tokenized["test"])
print("Test metrics:")
for k, v in test_metrics.items():
    print(f"  {k}: {v:.4f}" if isinstance(v, float) else f"  {k}: {v}")

pred_output = trainer.predict(tokenized["test"])
y_pred = np.argmax(pred_output.predictions, axis=-1)
y_true = pred_output.label_ids

print("\nClassification report:\n")
print(classification_report(y_true, y_pred, target_names=LABELS, digits=4))

Training Loss,Validation Loss,Epoch,Accuracy,F1 Macro,F1 Weighted
0.037463,0.154735,3,0.955043,0.954752,0.955020


Test metrics:
  eval_loss: 0.1547
  eval_accuracy: 0.9550
  eval_f1_macro: 0.9548
  eval_f1_weighted: 0.9550



Classification report:

              precision    recall  f1-score   support

    positive     0.9530    0.9638    0.9584       442
    negative     0.9574    0.9449    0.9511       381

    accuracy                         0.9550       823
   macro avg     0.9552    0.9543    0.9548       823
weighted avg     0.9551    0.9550    0.9550       823



## 6. Save & quick inference

In [8]:
save_path = os.path.join(OUTPUT_DIR, "best")
trainer.save_model(save_path)
tokenizer.save_pretrained(save_path)
print("Saved model + tokenizer to:", save_path)


@torch.no_grad()
def predict(headline: str, body: str = ""):
    model.eval()
    enc = tokenizer(
        headline, body or None,
        truncation="only_second", max_length=MAX_LENGTH, return_tensors="pt",
    ).to(model.device)
    probs = torch.softmax(model(**enc).logits, dim=-1)[0]
    return {id2label[i]: round(float(p), 4) for i, p in enumerate(probs)}


print(predict("gold prices surge to record high on safe-haven demand"))
print(predict("gold slumps as dollar strengthens and rate-hike fears grow"))

Writing model shards: 100%|██████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 1/1 [00:00<00:00,  2.26it/s]

Saved model + tokenizer to: goldbert-finetuned/best
{'positive': 0.9994, 'negative': 0.0006}
{'positive': 0.0046, 'negative': 0.9954}
